[Use PyTorch for all questions]

In [ ]:
import copy
import math
import random
import time
from collections import Counter
from pathlib import Path
from urllib.request import urlretrieve
from zipfile import ZipFile

import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.nn import functional as F

torch.manual_seed(0)
random.seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"
device

## Predicting the next word with an RNN

Our goal: read a text word by word, and at every position predict the **next word**.

```
input :  the   cat   sat   on    the
target:  cat   sat   on    the   mat
```

In the lecture, we built a recurrent cell from scratch (and saw why gradients vanish through time), and we saw why words are not fed to the network as one-hot vectors: each word is **embedded** (`nn.Embedding`), and the network predicts a small vector that a decoder turns into word scores. Here we use PyTorch's `nn.GRU` and `nn.Embedding`.

We will:
1. train the same RNN three times:
    - on top of the CBOW embedding of Session 8 (frozen),
    - on top of the official word2vec embedding (frozen),
    - end to end, learning its own embedding;
2. compare the performances, and the embeddings themselves.

### 0. Data

Same data as in Session 8: `text8`, the first $10^8$ characters of Wikipedia, cleaned (lowercase, no punctuation).
We keep the 10 000 most frequent words, every other word becomes `UNK`.

This time we keep a **validation set**: the last 10% of the text, never seen during training.

In [ ]:
if not Path("text8.zip").exists():
    urlretrieve("http://mattmahoney.net/dc/text8.zip", "text8.zip")
words = ZipFile("text8.zip").read("text8").decode().split()

counts = Counter(words)
itos = [word for word, _ in counts.most_common(10000)] + ["UNK"]
stoi = {word: i for i, word in enumerate(itos)}
UNK = stoi["UNK"]
vocab_size = len(itos)

ids = torch.tensor([stoi.get(word, UNK) for word in words])
split = int(0.9 * len(ids))
train_ids, val_ids = ids[:split], ids[split:]
print(f"{len(train_ids):,} training words, {len(val_ids):,} validation words, vocabulary of {vocab_size}")

A training example is a window of `seq_len + 1` consecutive words, cut at a random position:
the input `x` is the window without its last word, the target `y` is the window without its first word.
So `y[t]` is the word that comes right after `x[t]`.

In [ ]:
def get_batch(source, batch_size=64, seq_len=32):
    starts = torch.randint(0, len(source) - seq_len - 1, (batch_size,))
    windows = source[starts[:, None] + torch.arange(seq_len + 1)]
    x, y = windows[:, :-1], windows[:, 1:]  # y is x shifted by one word
    return x.to(device), y.to(device)

x, y = get_batch(train_ids, batch_size=2, seq_len=8)
for row_x, row_y in zip(x, y):
    print("x:", " ".join(itos[i] for i in row_x))
    print("y:", " ".join(itos[i] for i in row_y))
    print()

### 1. The language model

Implement a class `RNNLM(embedding, decoder, hidden_size=256, freeze=False)`:
- `embedding` is a `nn.Embedding(vocab_size, d)` (word index → vector of size `d`),
- `decoder` is a `nn.Linear(d, vocab_size)` (vector of size `d` → one score per word),
- in between: a `nn.GRU(d, hidden_size, batch_first=True)` and a `nn.Linear(hidden_size, d)`,
- if `freeze` is `True`, the parameters of `embedding` and `decoder` must not be trained (`requires_grad`).

`forward(x)` takes word indices of shape `[batch, seq_len]` and returns scores (logits) of shape `[batch, seq_len, vocab_size]`.

In [ ]:
def n_params(*modules):
    """Number of trainable parameters."""
    return sum(p.numel() for m in modules for p in m.parameters() if p.requires_grad)

Write a function `train_model(model, steps)` that trains `model` with Adam (learning rate 0.003) on `steps` batches from `get_batch(train_ids)`, with the cross-entropy loss between the scores and the targets **at every position**. Only give the trainable parameters to the optimizer, and clip the gradient norm to 1 (RNNs are prone to exploding gradients).
It should return the list of training losses.

*Hint:* `F.cross_entropy` wants scores of shape `[N, vocab_size]` and targets of shape `[N]`: reshape.

*On a laptop CPU, one training of 3000 steps takes about 10 minutes. If you have a GPU (or use Colab), it is much faster. If you are short on time, lower `STEPS` (but the end-to-end model needs the most steps: with fewer, it falls behind).*

To evaluate a language model, we use the validation cross-entropy, and the **perplexity** $= e^{\text{cross-entropy}}$:
it is the number of words the model hesitates between, on average. A model that predicts the uniform distribution has a perplexity equal to `vocab_size`; a perfect model has a perplexity of 1.
We also report the top-1 accuracy (is the most likely word the right one?).

In [ ]:
val_starts = torch.randint(0, len(val_ids) - 33, (1000,), generator=torch.Generator().manual_seed(1))
val_windows = val_ids[val_starts[:, None] + torch.arange(33)]  # always the same 1000 windows

@torch.no_grad()
def evaluate(model):
    total_ce, correct = 0.0, 0
    for k in range(0, len(val_windows), 100):
        x, y = val_windows[k:k + 100, :-1].to(device), val_windows[k:k + 100, 1:].to(device)
        logits = model(x)
        total_ce += F.cross_entropy(logits.reshape(-1, vocab_size), y.reshape(-1), reduction="sum").item()
        correct += (logits.argmax(-1) == y).sum().item()
    ce = total_ce / val_windows[:, 1:].numel()
    return {"cross-entropy": ce, "perplexity": math.exp(ce), "accuracy": correct / val_windows[:, 1:].numel()}

Before training anything: what perplexity does a model get if it ignores the context, and always predicts the frequency of each word in the training set (the *unigram* model)?
And what top-1 accuracy (which word would it always predict)?

*Your answer here.*

### 2. RNN on top of a frozen, pre-trained embedding

#### 2.a. The CBOW of Session 8

The cell below retrains the CBOW model of Session 8 (same model, same hyper-parameters), on the training part of the text only.
Instead of building multi-hot vectors with Python loops, it directly sums the columns of `encoder.weight`: same computation, but about a minute instead of half an hour.

In [ ]:
class CBOW(nn.Module):  # the model of Session 8
    def __init__(self, vocab_size, intermediate_size):
        super().__init__()
        self.encoder = nn.Linear(vocab_size, intermediate_size, bias=False)
        self.decoder = nn.Linear(intermediate_size, vocab_size)

    def forward(self, x):
        # decoder(encoder(multi_hot(x))), without building the multi-hot vectors
        return self.decoder(self.encoder.weight[:, x].sum(-1).T)

def cbow_batch(source, batch_size=256):
    i = torch.randint(2, len(source) - 2, (batch_size,))
    x = torch.stack([source[i - 2], source[i - 1], source[i + 1], source[i + 2]], dim=1)
    return x.to(device), source[i].to(device)

torch.manual_seed(0)
cbow = CBOW(vocab_size, 100).to(device)
optimizer = torch.optim.Adam(cbow.parameters(), lr=0.003)
start = time.time()
for step in range(30 * 132):  # 30 epochs of 132 batches, as in Session 8
    x, y = cbow_batch(train_ids)
    loss = F.cross_entropy(cbow(x), y)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
print(f"CBOW trained in {time.time() - start:.0f}s, last loss {loss.item():.3f}")

Build a `RNNLM` whose embedding is the (frozen) CBOW encoder and whose decoder is the (frozen) CBOW decoder. Train it and evaluate it.

*Hints:* the embedding of word `j` is column `j` of `cbow.encoder.weight`; `nn.Embedding.from_pretrained` builds an embedding from a matrix of shape `[vocab_size, d]`. Use `copy.deepcopy` on the decoder so that the CBOW model itself is left untouched.

#### 2.b. The official word2vec

Our CBOW is a simplified version of word2vec (Mikolov et al., 2013). The official implementation adds a few tricks: **negative sampling** (instead of a softmax over the whole vocabulary, only push the right word up and a few random words down), **subsampling** of very frequent words, and it goes through every position of the text.
`gensim` provides a faithful (and fast) re-implementation. We train it on the same training text, with the same embedding size (100) and the same window (2 words on each side) as our CBOW.

In [ ]:
%pip install gensim

In [ ]:
from gensim.models import Word2Vec

train_words = [itos[i] for i in train_ids.tolist()]  # rare words are already replaced by UNK
sentences = [train_words[k:k + 10000] for k in range(0, len(train_words), 10000)]
start = time.time()
w2v = Word2Vec(sentences, vector_size=100, window=2, sg=0, negative=5, min_count=1, epochs=5, workers=4, seed=0)
print(f"word2vec trained in {time.time() - start:.0f}s")

word2vec also learns two matrices: the input vectors `w2v.wv.vectors` (the embedding, one row per word) and the output vectors `w2v.syn1neg` (the decoder, one row per word). Their rows are in gensim's own order: the row of word `w` is `w2v.wv.key_to_index[w]`.

The decoder has no bias. With negative sampling, one can show that the score $u \cdot v_w$ approximates $\log \frac{p(w \mid \text{context})}{p(w)}$ (up to a constant), so we add the (fixed, not learned) bias $\log p(w)$, where $p(w)$ is the frequency of word $w$ in the training set (the unigram distribution computed above), to get scores that behave like log-probabilities.

Build the frozen embedding and the frozen decoder from word2vec, **in our vocabulary order**, and use them in a `RNNLM`. Train it and evaluate it.

### 3. Training end to end

Now, the same architecture, but with a random embedding and a random decoder, trained together with the RNN: the network learns its own embedding, only from the next-word prediction task.
Train it and evaluate it.

*Careful with the initialisation:* `nn.Embedding` initialises its weights with $\mathcal{N}(0, 1)$, about 100 times larger than `nn.Linear(vocab_size, 100)` (which uses $\pm 1/\sqrt{\text{vocab\_size}}$). A word's vector is only updated when the word appears in the batch, so with Adam's small steps a large random initialisation stays dominant for a long time, and the learned embedding looks random. Re-initialise it with `nn.init.normal_(embedding.weight, std=0.01)`.

Print a table with, for each model, the number of trainable parameters, the validation perplexity and the accuracy. Plot the training losses of the three models (smooth them with a moving average of 50 steps).

What do you conclude? Is it worth pre-training the embedding?

*Your answer here.*

### 4. Is the learned embedding similar to word2vec's?

The end-to-end model was never asked to learn a meaningful embedding: only to predict the next word. Let's look at what it learned.

Get the three embedding matrices, of shape `[vocab_size, 100]`: `E_cbow`, `E_w2v` and `E_e2e`.
Then write a function `nearest(E, word, k=8)` returning the `k` words whose embeddings have the highest cosine similarity with the embedding of `word` (excluding `word` itself), and print them side by side for the three embeddings, for a few words.

To quantify it, write `neighbour_overlap(E1, E2, k=10, n_words=2000)`: for each of the `n_words` most frequent words, take its `k` nearest neighbours in `E1` and in `E2`, and compute the fraction of neighbours they have in common; return the average over the words.

Compute it between the end-to-end embedding and word2vec, between CBOW and word2vec, and, as references:
- between word2vec and a random embedding `torch.randn(vocab_size, 100)` (what you get by chance);
- between word2vec and **another word2vec**, trained exactly the same way but with `seed=1` (how stable is the embedding itself?).

Finally, the famous property of word2vec: **analogies**. *paris* is to *france* what *rome* is to *italy*, so we hope that $v_{\text{france}} - v_{\text{paris}} + v_{\text{rome}}$ is close to $v_{\text{italy}}$.

The cell below loads the analogy test set of the word2vec paper (questions `a b c d` meaning "a is to b what c is to d"), keeping the questions whose 4 words are in our vocabulary.
Write `analogy_accuracy(E, questions)`: for each question, predict the word whose embedding has the highest cosine similarity with $v_b - v_a + v_c$ (excluding $a$, $b$ and $c$), and return the fraction of correct predictions. Compute it for the three embeddings.

In [ ]:
from gensim.test.utils import datapath

questions = []
for line in open(datapath("questions-words.txt")):
    q = line.lower().split()
    if not line.startswith(":") and all(word in stoi for word in q):
        questions.append([stoi[word] for word in q])
questions = torch.tensor(questions)
print(len(questions), "questions, e.g.", [itos[i] for i in questions[0]], [itos[i] for i in questions[-1]])

What do you conclude: is the embedding learned end to end similar to word2vec's?
Look again at the neighbours of *cold* in the end-to-end embedding (and in CBOW): why are *trojan*, *korean*, *civil* or *nuclear* there?

*Your answer here.*

### 5. Bonus: generate text

A language model gives a probability for the next word: we can **sample** from it, append the sampled word to the input, and repeat.
Write `generate(model, prompt, n_words=20, temperature=0.8)`, where the scores are divided by `temperature` before the softmax. Try different temperatures: what happens when it is close to 0? Large?

*Your answer here.*